In [ ]:
import pandas as pd
import os

def transform_staff_data(input_file, output_file=None):
    """
    Transform staff data CSV file to:
    1. Standardize sex descriptors (F->Female, M->Male, blank/U->Not selected)
    2. Set identification system to 'State Education Agency'
    3. Map race values to standard descriptors
    
    Parameters:
    - input_file: Path to the input CSV file
    - output_file: Path to the output CSV file (if None, will use input_file with '_transformed' suffix)
    """
    if output_file is None:
        base, ext = os.path.splitext(input_file)
        output_file = f"{base}_transformed{ext}"
    
    print(f"Reading data from {input_file}...")
    
    # Try different encodings
    encodings_to_try = ['utf-8', 'latin-1', 'ISO-8859-1', 'cp1252']
    
    for encoding in encodings_to_try:
        try:
            df = pd.read_csv(input_file, encoding=encoding)
            print(f"Successfully read CSV with encoding: {encoding}")
            break
        except UnicodeDecodeError:
            if encoding == encodings_to_try[-1]:
                print(f"Failed to read file with any encoding. Please check the file format.")
                return
            continue
    
    # Define column names
    sex_col = '/ed-fi/staffs/sexDescriptor'
    id_system_col = '/ed-fi/staffs/identificationCodes[0].staffIdentificationSystemDescriptor'
    race_col = '/ed-fi/staffs/races[n].raceDescriptor'
    
    # 1. Update sex descriptor
    if sex_col in df.columns:
        print(f"Updating sex descriptor values...")
        df[sex_col] = df[sex_col].apply(
            lambda x: 'Female' if x == 'F' else (
                'Male' if x == 'M' else (
                    'Not selected' if pd.isna(x) or x == 'U' or x == '' else x
                )
            )
        )
    else:
        print(f"Warning: Sex descriptor column '{sex_col}' not found.")
    
    # 2. Update identification system
    if id_system_col in df.columns:
        print(f"Setting identification system to 'State Education Agency'...")
        df[id_system_col] = 'State Education Agency'
    else:
        print(f"Warning: Identification system column '{id_system_col}' not found.")
    
    # 3. Update race values
    if race_col in df.columns:
        print(f"Updating race descriptor values...")
        
        race_mapping = {
            'White': 'White',
            'Black': 'Black - African American',
            'Latinx': 'Hispanic',
            'Hispanic': 'Hispanic',
            'Asian': 'Asian',
            'Other': 'Other',
            'American Indian': 'American Indian - Alaska Native',
            'Native Hawaiian': 'Native Hawaiian - Pacific Islander',
        }
        
        # Function to map race values
        def map_race(value):
            if pd.isna(value) or value == '':
                return 'Choose Not to Respond'
            
            # Check for exact matches
            if value in race_mapping:
                return race_mapping[value]
            
            # Check for partial matches
            for key, mapped_value in race_mapping.items():
                if isinstance(value, str) and key in value:
                    return mapped_value
            
            # Default to 'No data' for unmatched values
            return 'Choose Not to Respond'
        
        df[race_col] = df[race_col].apply(map_race)
    else:
        print(f"Warning: Race descriptor column '{race_col}' not found.")
    
    #Convert the Primary Email Address Indicator to a boolean
    # Save the transformed data
    print(f"Saving transformed data to {output_file}...")
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Done! Transformed file saved to {output_file}")
    
    return df



In [6]:

input_file = '../../data/staffs/staffs.csv'
output_file = '../../data/staffs/staffs_transformed.csv'
transform_staff_data(input_file, output_file)

Reading data from ../../data/staffs/staffs.csv...
Successfully read CSV with encoding: utf-8
Updating sex descriptor values...
Setting identification system to 'State Education Agency'...
Updating race descriptor values...
Saving transformed data to ../../data/staffs/staffs_transformed.csv...
Done! Transformed file saved to ../../data/staffs/staffs_transformed.csv


,SchoolYear,/ed-fi/staffs/staffUniqueId,/ed-fi/staffs/birthDate,/ed-fi/staffs/citizenshipStatusDescriptor,/ed-fi/staffs/electronicMails[0].electronicMailTypeDescriptor,/ed-fi/staffs/electronicMails[0].electronicMailAddress,/ed-fi/staffs/electronicMails[0].doNotPublishIndicator,/ed-fi/staffs/electronicMails[0].primaryEmailAddressIndicator,/ed-fi/staffs/firstName,/ed-fi/staffs/identificationCodes[0].staffIdentificationSystemDescriptor,/ed-fi/staffs/identificationCodes[0].assigningOrganizationIdentificationCode,/ed-fi/staffs/lastSurname,/ed-fi/staffs/races[n].raceDescriptor,/ed-fi/staffs/sexDescriptor
0,2003,829,02/21/1932,NaN,Work,000829@boston.k12.ma.us,NaN,1,Claire,State Education Agency,NaN,Pettingell,White,Female
1,2004,829,02/21/1932,NaN,Work,000829@boston.k12.ma.us,NaN,1,Claire,State Education Agency,NaN,Pettingell,White,Female
2,2007,1866,03/10/1930,NaN,Work,001866@boston.k12.ma.us,NaN,1,Nina,State Education Agency,NaN,Kearney,White,Female
3,2008,1866,03/10/1930,NaN,Work,001866@boston.k12.ma.us,NaN,1,Nina,State Education Agency,NaN,Kearney,White,Female
4,1999,7305,06/17/1956,NaN,Work,007305@boston.k12.ma.us,NaN,1,Loretta,State Education Agency,50791916.0,Murphy,White,Female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258549,2023,178949,11/17/1990,NaN,Work,178949@boston.k12.ma.us,NaN,1,Elizabeth,State Education Agency,NaN,Delacruz,Hispanic,Female
258550,2023,179069,01/03/2001,NaN,Work,179069@boston.k12.ma.us,NaN,1,Yang,State Education Agency,NaN,Yang,Asian,Female
258551,2023,179284,10/25/1968,NaN,Work,179284@boston.k12.ma.us,NaN,1,Eduarda,State Education Agency,NaN,Pualino De Arias,Hispanic,Female
258552,2023,179285,10/20/1967,NaN,Work,179285@boston.k12.ma.us,NaN,1,Dilenia,State Education Agency,NaN,Pena,Hispanic,Female
